# Release16 State-Head Revision Handoff

Purpose: convert the latest `release16_failedfold_state_head_checkpoint_bc_screen_v1` failure into a concrete next target/objective revision before any retained actor training. This is a diagnostic handoff only; it does not promote, retain, or train an actor.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "SPEC.md").exists())
CARD = ROOT / "runtime/model_selection/card/release16_state_head_revision_handoff_v1"
SOURCE = ROOT / "runtime/model_selection/card/release16_failedfold_state_head_checkpoint_bc_screen_v1"
RETENTION_GATE = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"
CARD.mkdir(parents=True, exist_ok=True)

def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path: Path, payload):
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

source_summary = load_json(SOURCE / "summary.json")
gate_payload = load_json(RETENTION_GATE)
gate_summary = gate_payload.get("summary", gate_payload)
TARGET_REL = float(source_summary.get("target_relative_return_pp", 30.0))
TARGET_DD = float(source_summary.get("target_max_dd_pct", gate_summary.get("max_walkforward_drawdown_pct", 20.0)))
STRICT_DD = 15.0
REQUIRED_WINDOWS = list(gate_summary.get("required_wf_windows", ["wf_2020", "wf_2022", "wf_2023_2024"]))

checkpoint = pd.read_csv(SOURCE / "checkpoint_selection_by_fold.csv")
delta = pd.read_csv(SOURCE / "state_head_vs_drawdown_upside_checkpoint_delta.csv")
bc_slices = pd.read_csv(SOURCE / "bc_slice_metrics.csv")
print({"target_relative_return_pp": TARGET_REL, "target_max_dd_pct": TARGET_DD, "rows": len(checkpoint)})

{'target_relative_return_pp': 30.0, 'target_max_dd_pct': 20.0, 'rows': 3}


In [2]:
matrix = checkpoint.merge(delta, on="fold", how="left")
for col in ["selected_relative_return", "selected_max_dd", "selected_avg_pos", "selected_return", "relative_return_delta_pp", "max_dd_delta_pct", "avg_pos_delta"]:
    matrix[col] = pd.to_numeric(matrix[col], errors="coerce")

matrix["relative_gap_to_target_pp"] = (TARGET_REL - matrix["selected_relative_return"]).clip(lower=0)
matrix["drawdown_gap_to_target_pp"] = (matrix["selected_max_dd"] - TARGET_DD).clip(lower=0)
matrix["strict_drawdown_gap_pp"] = (matrix["selected_max_dd"] - STRICT_DD).clip(lower=0)
matrix["metric_gate_pass_20dd"] = (matrix["selected_relative_return"] >= TARGET_REL) & (matrix["selected_max_dd"] < TARGET_DD)
matrix["strict_gate_pass_15dd"] = (matrix["selected_relative_return"] >= TARGET_REL) & (matrix["selected_max_dd"] <= STRICT_DD)

def classify(row):
    rel_fail = row["relative_gap_to_target_pp"] > 0
    dd_fail = row["drawdown_gap_to_target_pp"] > 0
    slice_fail = not bool(row.get("selected_bc_slice_gate", False))
    if rel_fail and dd_fail:
        return "joint_tradeoff_state_head_overexposure"
    if dd_fail:
        return "drawdown_drift_after_state_head_lift"
    if rel_fail:
        return "underparticipation_or_wrong_state_lift"
    if slice_fail:
        return "metric_pass_but_bc_slice_gate_failed"
    return "passed_metric_gate"

matrix["failure_type"] = matrix.apply(classify, axis=1)
matrix["target_revision_need"] = matrix.apply(
    lambda row: (
        "risk_gate_state_head_lift_and_penalize_drawdown_drift"
        if row["drawdown_gap_to_target_pp"] > 0
        else "repair_state_head_slice_fit_before_training"
        if not bool(row.get("selected_bc_slice_gate", False))
        else "preserve_metric_behavior"
    ),
    axis=1,
)
fold_failure_matrix = matrix[[
    "fold",
    "selected_candidate",
    "selected_relative_return",
    "selected_max_dd",
    "selected_avg_pos",
    "selected_return",
    "relative_gap_to_target_pp",
    "drawdown_gap_to_target_pp",
    "strict_drawdown_gap_pp",
    "metric_gate_pass_20dd",
    "strict_gate_pass_15dd",
    "selected_target_gate",
    "selected_drift_gate",
    "selected_bc_slice_gate",
    "selected_candidate_gate",
    "relative_return_delta_pp",
    "max_dd_delta_pct",
    "avg_pos_delta",
    "failure_type",
    "target_revision_need",
]]
fold_failure_matrix.to_csv(CARD / "fold_failure_matrix.csv", index=False)
fold_failure_matrix

,fold,selected_candidate,selected_relative_return,selected_max_dd,selected_avg_pos,selected_return,relative_gap_to_target_pp,drawdown_gap_to_target_pp,strict_drawdown_gap_pp,metric_gate_pass_20dd,strict_gate_pass_15dd,selected_target_gate,selected_drift_gate,selected_bc_slice_gate,selected_candidate_gate,relative_return_delta_pp,max_dd_delta_pct,avg_pos_delta,failure_type,target_revision_need
0,wf_2020,bc_initialization,27.840731,26.374299,0.476405,85.934302,2.159269,6.374299,11.374299,False,False,False,True,False,False,28.267784,7.047392,0.173481,joint_tradeoff_state_head_overexposure,risk_gate_state_head_lift_and_penalize_drawdow...
1,wf_2022,bc_initialization,72.363307,15.487774,0.087614,-14.224764,0.000000,0.000000,0.487774,True,False,True,True,False,False,0.648327,-0.129269,0.006230,metric_pass_but_bc_slice_gate_failed,repair_state_head_slice_fit_before_training
2,wf_2023_2024,bc_initialization,-16.012653,30.559989,0.309119,-16.369288,46.012653,10.559989,15.559989,False,False,False,True,False,False,-19.004371,16.935372,0.092297,joint_tradeoff_state_head_overexposure,risk_gate_state_head_lift_and_penalize_drawdow...


In [3]:
slice_focus = bc_slices[bc_slices["fold"].isin(REQUIRED_WINDOWS)].copy()
slice_focus["accuracy"] = pd.to_numeric(slice_focus["accuracy"], errors="coerce")
slice_focus["samples"] = pd.to_numeric(slice_focus["samples"], errors="coerce")
required_slice_summary = (
    slice_focus[slice_focus["slice"].isin(["all", "failedfold_state_head_target_state", "drawdown_control_state", "is_downtrend_state"])]
    .sort_values(["fold", "slice"])
    [["fold", "slice", "samples", "accuracy", "label_unique_actions", "predicted_unique_actions", "label_dominant_share", "predicted_dominant_share"]]
)
required_slice_summary.to_csv(CARD / "bc_slice_gate_focus.csv", index=False)
required_slice_summary.head(12)

,fold,slice,samples,accuracy,label_unique_actions,predicted_unique_actions,label_dominant_share,predicted_dominant_share
0,wf_2020,all,1905,0.604724,5,2,0.525984,0.869291
1,wf_2020,drawdown_control_state,627,0.551834,5,2,0.518341,0.961722
3,wf_2020,failedfold_state_head_target_state,906,0.772627,5,2,0.652318,0.790287
4,wf_2020,is_downtrend_state,627,0.551834,5,2,0.518341,0.961722
7,wf_2022,all,253,0.877470,5,5,0.418972,0.478261
8,wf_2022,drawdown_control_state,0,NaN,0,0,1.000000,1.000000
10,wf_2022,failedfold_state_head_target_state,0,NaN,0,0,1.000000,1.000000
11,wf_2022,is_downtrend_state,118,0.949153,5,5,0.457627,0.432203
14,wf_2023_2024,all,756,0.582011,5,3,0.324074,0.685185
15,wf_2023_2024,drawdown_control_state,0,NaN,0,0,1.000000,1.000000


In [4]:
revision_recipe = pd.DataFrame([
    {
        "recipe_step": "risk_gated_state_head_lift",
        "applies_to_folds": "wf_2020,wf_2023_2024",
        "evidence": "state-head raised avg_pos and MaxDD versus drawdown-upside baseline in the failed folds",
        "implementation_target": "Gate positive state-head lifts behind drawdown/volshock/downtrend safety state; avoid lifting when expected MaxDD drift is positive.",
        "acceptance_gate": "all wf folds relative_return >= 30pp and MaxDD < 20%; no failed fold has positive MaxDD delta from the drawdown-upside baseline",
    },
    {
        "recipe_step": "drawdown_drift_checkpoint_objective",
        "applies_to_folds": "wf_2020,wf_2023_2024",
        "evidence": "wf_2020 is near the 30pp relative target but MaxDD is 26.37%; wf_2023_2024 flips to negative relative return with 30.56% MaxDD",
        "implementation_target": "Add checkpoint/reward loss terms for fold-local drawdown drift and trend-preservation, not a longer PPO continuation of this actor.",
        "acceptance_gate": "checkpoint replay selects a non-drifting candidate; candidate_gate and metric gate pass on wf_2020/wf_2022/wf_2023_2024",
    },
    {
        "recipe_step": "state_head_slice_fit_repair",
        "applies_to_folds": "wf_2020,wf_2022,wf_2023_2024",
        "evidence": "selected_bc_slice_gate is false in every fold, including wf_2022 where headline metrics pass",
        "implementation_target": "Repair sparse state-head target semantics or model objective before any retained actor training.",
        "acceptance_gate": "state-head required slices pass before PPO; otherwise keep actor_retention_allowed=false",
    },
])
revision_recipe.to_csv(CARD / "revision_recipe.csv", index=False)
revision_recipe

,recipe_step,applies_to_folds,evidence,implementation_target,acceptance_gate
0,risk_gated_state_head_lift,"wf_2020,wf_2023_2024",state-head raised avg_pos and MaxDD versus dra...,Gate positive state-head lifts behind drawdown...,all wf folds relative_return >= 30pp and MaxDD...
1,drawdown_drift_checkpoint_objective,"wf_2020,wf_2023_2024",wf_2020 is near the 30pp relative target but M...,Add checkpoint/reward loss terms for fold-loca...,checkpoint replay selects a non-drifting candi...
2,state_head_slice_fit_repair,"wf_2020,wf_2022,wf_2023_2024","selected_bc_slice_gate is false in every fold,...",Repair sparse state-head target semantics or m...,state-head required slices pass before PPO; ot...


In [5]:
failed_folds = fold_failure_matrix.loc[~fold_failure_matrix["metric_gate_pass_20dd"], "fold"].tolist()
metric_gate_pass_count = int(fold_failure_matrix["metric_gate_pass_20dd"].sum())
strict_gate_pass_count = int(fold_failure_matrix["strict_gate_pass_15dd"].sum())
bc_slice_gate_count = int(fold_failure_matrix["selected_bc_slice_gate"].fillna(False).astype(bool).sum())
worst_dd = float(fold_failure_matrix["selected_max_dd"].max())
min_rel = float(fold_failure_matrix["selected_relative_return"].min())
worst_rel_fold = str(fold_failure_matrix.sort_values("selected_relative_return").iloc[0]["fold"])
worst_dd_fold = str(fold_failure_matrix.sort_values("selected_max_dd", ascending=False).iloc[0]["fold"])

summary = {
    "experiment": "release16_state_head_revision_handoff_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "role": "diagnose_target_objective_revision",
    "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
    "direct_comparison_baseline": "release16_failedfold_state_head_checkpoint_bc_screen_v1",
    "source_card": "release16_failedfold_state_head_checkpoint_bc_screen_v1",
    "reference_baseline": source_summary.get("reference_baseline"),
    "decision": "state_head_revision_handoff_no_training_no_actor_retention",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_from_this_card": False,
    "target_revision_ready": False,
    "next_training_allowed_after_target_revision": False,
    "next_mechanism": "build_risk_gated_state_head_target_or_drawdown_drift_checkpoint_objective_then_rerun_official_three_fold_replay",
    "changed_layer": ["model_selection", "target_revision", "checkpoint_objective"],
    "target_relative_return_pp": TARGET_REL,
    "target_max_dd_pct": TARGET_DD,
    "strict_max_dd_context_pct": STRICT_DD,
    "failed_folds": failed_folds,
    "metric_gate_pass_count": metric_gate_pass_count,
    "strict_gate_pass_count": strict_gate_pass_count,
    "bc_slice_gate_count": bc_slice_gate_count,
    "worst_wf_max_drawdown_pct": round(worst_dd, 6),
    "min_wf_relative_return_pp": round(min_rel, 6),
    "worst_relative_fold": worst_rel_fold,
    "worst_drawdown_fold": worst_dd_fold,
    "primary_blocker": "walkforward_relative_return_gap",
    "metrics": [
        {
            "window": str(row.fold),
            "split_type": "walk_forward_test",
            "return_pct": float(row.selected_return),
            "relative_return_pct": float(row.selected_relative_return),
            "max_dd_pct": float(row.selected_max_dd),
            "avg_pos": float(row.selected_avg_pos),
        }
        for row in fold_failure_matrix.itertuples(index=False)
    ],
    "conclusion": [
        f"State-head screen passes 0/3 candidate gates and {metric_gate_pass_count}/3 headline 20% drawdown gates; no actor retention.",
        f"wf_2020 nearly reaches the relative-return target ({fold_failure_matrix.loc[fold_failure_matrix.fold == 'wf_2020', 'selected_relative_return'].iloc[0]:.2f}pp) but MaxDD is {fold_failure_matrix.loc[fold_failure_matrix.fold == 'wf_2020', 'selected_max_dd'].iloc[0]:.2f}%.",
        f"wf_2023_2024 is the worst fold: relative {fold_failure_matrix.loc[fold_failure_matrix.fold == 'wf_2023_2024', 'selected_relative_return'].iloc[0]:.2f}pp and MaxDD {fold_failure_matrix.loc[fold_failure_matrix.fold == 'wf_2023_2024', 'selected_max_dd'].iloc[0]:.2f}%.",
        "Next step is target/model-objective revision with explicit drawdown-drift control, not longer PPO or actor retention.",
    ],
    "sidecars": [
        "fold_failure_matrix.csv",
        "bc_slice_gate_focus.csv",
        "revision_recipe.csv",
        "selection.json",
        "summary.json",
    ],
}
selection = dict(summary)
selection["selected_next_action"] = summary["next_mechanism"]
write_json(CARD / "summary.json", summary)
write_json(CARD / "selection.json", selection)
summary

{'experiment': 'release16_state_head_revision_handoff_v1',
 'created_at': '2026-06-14T23:32:02.304169+00:00',
 'role': 'diagnose_target_objective_revision',
 'canonical_baseline': 'soxlbacktest_policy_reimplementation_v1',
 'direct_comparison_baseline': 'release16_failedfold_state_head_checkpoint_bc_screen_v1',
 'source_card': 'release16_failedfold_state_head_checkpoint_bc_screen_v1',
 'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02',
 'decision': 'state_head_revision_handoff_no_training_no_actor_retention',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'target_revision_ready': False,
 'next_training_allowed_after_target_revision': False,
 'next_mechanism': 'build_risk_gated_state_head_target_or_drawdown_drift_checkpoint_objective_then_rerun_official_three_fold_replay',
 'changed_layer': ['model_selection',
  'target_revision',
  'checkpoint_objective'],
 'target_relative_return_pp': 3

## Conclusion

This notebook is expected to end with the generated `summary.json`; rerun it after changing the state-head target or checkpoint objective.

In [6]:
print("decision=", summary["decision"])
print("next=", summary["next_mechanism"])
print("failed_folds=", ",".join(summary["failed_folds"]))
print("metric_gate_pass_count=", summary["metric_gate_pass_count"], "bc_slice_gate_count=", summary["bc_slice_gate_count"])

decision= state_head_revision_handoff_no_training_no_actor_retention
next= build_risk_gated_state_head_target_or_drawdown_drift_checkpoint_objective_then_rerun_official_three_fold_replay
failed_folds= wf_2020,wf_2023_2024
metric_gate_pass_count= 1 bc_slice_gate_count= 0
